# 08. Tìm cấu hình chia tập đạt tiêu chí 1.5 (chỉ dựa trên đếm nhãn)

Với mỗi bộ: chronological, và blocked với 10 / 20 / 50 / 100 / 200 khối, chia theo số cửa sổ hoặc theo số node. Tiêu chí: mỗi lớp chính ≥ 100 node ở train, ≥ 30 ở val và test; tỉ lệ node train/val/test trong 70/15/15 ± 3 điểm %. Không dùng kết quả mô hình để chọn.

In [ ]:
import glob, os, subprocess, sys, time

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-6000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/tmp/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])

In [ ]:
JOBS = [("nf_ton_iot_v3", "NF-ToN-IoT-v3.csv", 5), ("nf_cicids2018_v3", "NF-CICIDS2018-v3.csv", 5),
        ("nf_ton_iot_v2", "NF-ToN-IoT-v2.csv", 1000)]
for ds, fn, w in JOBS:
    csv = next(glob.iglob(f"/kaggle/input/**/{fn}", recursive=True))
    t0 = time.time()
    print("=====", ds, csv)
    run([sys.executable, "-I", "scripts/split_search.py", ds, csv, str(w), str(w),
         f"/kaggle/working/split_search_{ds}.json"], cwd=REPO)
    print(f"{ds} xong sau {time.time() - t0:.0f} s")